# Unidad 8: Structured Streaming

**Tecnicatura en Datos – Apache Spark (Databricks)**  
Notebook de práctica — `spark` ya está disponible en el entorno

---

## Contenido

1. Batch vs Streaming: diferencias y casos de uso
2. Modelo de tabla infinita en Structured Streaming
3. Fuente Rate: generador de datos para pruebas
4. Ventanas de tiempo: tumbling y sliding
5. Watermarks: datos tardíos
6. Output modes: append, complete, update
7. `foreachBatch`: lógica personalizada
8. Práctica guiada: pipeline de alertas

## Configuración

In [ ]:
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Streaming activos al inicio: {len(spark.streams.active)}")

---
## 1. Batch vs Streaming

| Aspecto | Batch | Streaming |
|---------|-------|----------|
| Datos | Finitos | Infinitos (llegan continuamente) |
| Trigger | Programado (cron) | Continuo o micro-lotes |
| Latencia | Minutos a horas | Milisegundos a segundos |
| API | `spark.read` | `spark.readStream` |

**Structured Streaming** modela el stream como una **tabla que crece indefinidamente**. El desarrollador describe el resultado, Spark lo ejecuta incrementalmente.

---
## 2. Fuente Rate: primer stream

### Ejemplo 1 — Simple: stream con Rate source y sink a Delta

In [ ]:
import time
from pyspark.sql.functions import col, when

# Limpiar directorio previo
dbutils.fs.rm("/tmp/u08_stream_simple/", recurse=True)
dbutils.fs.rm("/tmp/u08_ckpt_simple/",  recurse=True)

# Fuente Rate: genera filas a 10 por segundo
# Cada fila tiene: timestamp (marca de tiempo), value (entero incremental)
stream_rate = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
    # Enriquecer con columnas de negocio simuladas
    .withColumn("pais",  when(col("value") % 3 == 0, "AR")
                         .when(col("value") % 3 == 1, "MX")
                         .otherwise("CL"))
    .withColumn("monto", ((col("value") * 37.7) % 5000 + 100).cast("double"))
)

print("Schema del stream:")
stream_rate.printSchema()

# Escribir a Delta (sink persistente)
query = (stream_rate.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/u08_ckpt_simple/")
    .start("/tmp/u08_stream_simple/")
)

time.sleep(12)   # Dejar correr 12 segundos
query.stop()

# Verificar resultado
df_resultado = spark.read.format("delta").load("/tmp/u08_stream_simple/")
print(f"\nFilas escritas en 12 segundos: {df_resultado.count()}")
df_resultado.groupBy("pais").count().show()

**Resultado esperado:**
```
Schema del stream:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- pais: string (nullable = true)
 |-- monto: double (nullable = true)

Filas escritas en 12 segundos: ~120

+----+-----+
|pais|count|
+----+-----+
|  AR|   40|
|  MX|   40|
|  CL|   40|
+----+-----+
```
> El `checkpointLocation` permite reiniciar el stream desde donde quedó sin reprocesar datos ya procesados.

---
## 3. Ventanas de tiempo

### Ejemplo 2 — Medio: Tumbling Windows (ventanas fijas sin solapamiento)

In [ ]:
from pyspark.sql.functions import window, col, sum as spark_sum, count

dbutils.fs.rm("/tmp/u08_ckpt_ventana/", recurse=True)

stream_tx = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
    .withColumn("pais",  when(col("value") % 4 == 0, "AR")
                         .when(col("value") % 4 == 1, "MX")
                         .when(col("value") % 4 == 2, "CL")
                         .otherwise("BR"))
    .withColumn("monto", ((col("value") * 53.3) % 4000 + 200).cast("double"))
)

# Tumbling window: ventanas de 5 segundos sin solapamiento
# |----5s----|----5s----|----5s----|
ventas_por_ventana = (stream_tx
    .groupBy(
        window(col("timestamp"), "5 seconds"),  # ventana de 5 segundos
        col("pais")
    )
    .agg(
        count("value").alias("num_tx"),
        spark_sum("monto").alias("monto_total")
    )
    .select(
        col("window.start").alias("inicio"),
        col("window.end").alias("fin"),
        "pais", "num_tx", "monto_total"
    )
)

query_ventana = (ventas_por_ventana.writeStream
    .outputMode("complete")   # complete = re-emite toda la tabla en cada micro-lote
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "/tmp/u08_ckpt_ventana/")
    .start()
)

time.sleep(18)
query_ventana.stop()
print("Query detenido")

**Resultado esperado (en consola del notebook):**
```
-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+----+------+------------+
|             inicio|                fin|pais|num_tx| monto_total|
+-------------------+-------------------+----+------+------------+
|2024-01-15 10:00:00|2024-01-15 10:00:05|  AR|    25|   52375.0  |
|2024-01-15 10:00:00|2024-01-15 10:00:05|  MX|    25|   61200.0  |
...
```
> Con `outputMode("complete")` el sink recibe toda la tabla en cada micro-lote. Útil cuando el resultado final depende de todos los datos acumulados.

### Ejemplo 3 — Avanzado: Sliding Windows + Watermark para datos tardíos

In [ ]:
dbutils.fs.rm("/tmp/u08_ckpt_sliding/", recurse=True)

stream_eventos = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 15)
    .load()
    .withColumn("evento_tipo", when(col("value") % 3 == 0, "click")
                                .when(col("value") % 3 == 1, "compra")
                                .otherwise("vista"))
)

# Watermark: tolerar datos que lleguen hasta 10 segundos tarde
# Sliding window: ventana de 10 segundos que avanza cada 5 segundos
# |----10s----|
#      |----10s----|
#           |----10s----|
eventos_sliding = (stream_eventos
    .withWatermark("timestamp", "10 seconds")   # tolerar hasta 10s de retraso
    .groupBy(
        window(col("timestamp"), "10 seconds", "5 seconds"),  # sliding window
        col("evento_tipo")
    )
    .count()
    .select(
        col("window.start").alias("inicio"),
        col("window.end").alias("fin"),
        "evento_tipo", "count"
    )
)

query_sliding = (eventos_sliding.writeStream
    .outputMode("update")   # update: solo emite ventanas actualizadas
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "/tmp/u08_ckpt_sliding/")
    .start()
)

time.sleep(20)
query_sliding.stop()
print("Query detenido")

**Resultado esperado:**
```
Batch: 2
+-------------------+-------------------+-----------+-----+
|             inicio|                fin|evento_tipo|count|
+-------------------+-------------------+-----------+-----+
|2024-01-15 10:00:00|2024-01-15 10:00:10|      click|   50|
|2024-01-15 10:00:00|2024-01-15 10:00:10|     compra|   50|
|2024-01-15 10:00:05|2024-01-15 10:00:15|      click|   75|   ← ventana solapada
...
```
> Con `outputMode("update")` solo se emiten las ventanas que cambiaron en este micro-lote. Más eficiente que `complete` para datasets grandes.

---
## 4. Output Modes

| Modo | Cuándo usar |
|------|-------------|
| `append` | Sin agregaciones, o con watermark. Solo emite filas nuevas |
| `complete` | Con agregaciones SIN watermark. Re-emite toda la tabla |
| `update` | Con agregaciones. Solo emite filas actualizadas |

In [ ]:
# Ejemplo: append mode (sin agrupaciones)
dbutils.fs.rm("/tmp/u08_ckpt_append/", recurse=True)
dbutils.fs.rm("/tmp/u08_sink_append/", recurse=True)

stream_simple = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("tipo", when(col("value") % 2 == 0, "A").otherwise("B"))
)

# append: cada micro-lote agrega sus filas al sink
q_append = (stream_simple.writeStream
    .outputMode("append")
    .format("delta")
    .option("checkpointLocation", "/tmp/u08_ckpt_append/")
    .start("/tmp/u08_sink_append/")
)

time.sleep(8)
q_append.stop()

total = spark.read.format("delta").load("/tmp/u08_sink_append/").count()
print(f"Filas en sink append: {total}")
print("\nÚltimas métricas del query:")
import json
if q_append.lastProgress:
    prog = q_append.lastProgress
    print(f"  numInputRows:           {prog['numInputRows']}")
    print(f"  inputRowsPerSecond:     {prog['inputRowsPerSecond']:.1f}")
    print(f"  processedRowsPerSecond: {prog['processedRowsPerSecond']:.1f}")

**Resultado esperado:**
```
Filas en sink append: ~40

Últimas métricas del query:
  numInputRows:           5
  inputRowsPerSecond:     5.0
  processedRowsPerSecond: 120.3
```

---
## 5. foreachBatch: lógica personalizada por micro-lote

`foreachBatch` recibe cada micro-lote como un DataFrame normal. Permite escribir a múltiples destinos, aplicar lógica compleja, o llamar a APIs externas.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

dbutils.fs.rm("/tmp/u08_ckpt_foreach/",    recurse=True)
dbutils.fs.rm("/tmp/u08_normales/",         recurse=True)
dbutils.fs.rm("/tmp/u08_alertas/",          recurse=True)

# Acumulador para contar alertas
total_alertas = spark.sparkContext.accumulator(0)

def procesar_lote(df_lote, id_lote):
    """Procesa cada micro-lote: separa normales de alertas."""
    df_enriquecido = (df_lote
        .withColumn("monto", ((col("value") * 91.1) % 10000 + 50).cast("double"))
        .withColumn("alerta", col("monto") > 7000)
        .withColumn("procesado_en", current_timestamp())
        .withColumn("lote", lit(id_lote))
    )
    
    df_normales = df_enriquecido.filter(~col("alerta"))
    df_alertas  = df_enriquecido.filter(col("alerta"))
    
    # Escribir a dos destinos distintos
    df_normales.write.format("delta").mode("append").save("/tmp/u08_normales/")
    df_alertas.write.format("delta").mode("append").save("/tmp/u08_alertas/")
    
    n_alertas = df_alertas.count()
    total_alertas.add(n_alertas)
    print(f"Lote {id_lote}: {df_lote.count()} filas → {n_alertas} alertas")

stream_datos = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 30)
    .load()
)

query_foreach = (stream_datos.writeStream
    .foreachBatch(procesar_lote)
    .option("checkpointLocation", "/tmp/u08_ckpt_foreach/")
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(20)
query_foreach.stop()

print(f"\n=== Resumen ===")
print(f"Total alertas acumuladas: {total_alertas.value}")
print(f"Normales: {spark.read.format('delta').load('/tmp/u08_normales/').count()}")
print(f"Alertas:  {spark.read.format('delta').load('/tmp/u08_alertas/').count()}")

**Resultado esperado:**
```
Lote 0: 150 filas → 45 alertas
Lote 1: 150 filas → 43 alertas
Lote 2: 150 filas → 47 alertas
Lote 3: 150 filas → 44 alertas

=== Resumen ===
Total alertas acumuladas: 179
Normales: 421
Alertas:  179
```
> `foreachBatch` es la opción más flexible: podés escribir a múltiples destinos, llamar a APIs REST, o aplicar lógica que no soporta el sink estándar.

---
## 6. Monitoreo de queries activas

In [ ]:
dbutils.fs.rm("/tmp/u08_ckpt_mon/",  recurse=True)

# Iniciar un stream
q_mon = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 100)
    .load()
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/u08_ckpt_mon/")
    .start("/tmp/u08_mon_sink/")
)

time.sleep(5)

# Ver queries activos
print("Queries activos:", len(spark.streams.active))

# Estado del query
print("\nStatus:", q_mon.status)

# Métricas del último micro-lote
if q_mon.lastProgress:
    prog = q_mon.lastProgress
    print(f"\nMétricas del último lote:")
    print(f"  Filas de entrada:    {prog['numInputRows']}")
    print(f"  Filas/seg entrada:   {prog['inputRowsPerSecond']:.1f}")
    print(f"  Filas/seg procesado: {prog['processedRowsPerSecond']:.1f}")

q_mon.stop()
print("\nQuery detenido. Queries activos:", len(spark.streams.active))

**Resultado esperado:**
```
Queries activos: 1

Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}

Métricas del último lote:
  Filas de entrada:    100
  Filas/seg entrada:   100.0
  Filas/seg procesado: 856.4

Query detenido. Queries activos: 0
```

---
## 7. Práctica guiada: pipeline de alertas con ventanas

In [ ]:
from pyspark.sql.functions import window, col, when, avg, count, round as spark_round

dbutils.fs.rm("/tmp/u08_ckpt_practica/", recurse=True)

# Stream de transacciones simuladas
stream_practica = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 25)
    .load()
    .withColumn("cliente_id", (col("value") % 50).cast("int"))
    .withColumn("monto",      ((col("value") * 113.7) % 8000 + 100).cast("double"))
    .withColumn("pais",       when(col("value") % 3 == 0, "AR")
                              .when(col("value") % 3 == 1, "MX")
                              .otherwise("CL"))
)

# Calcular métricas por cliente en ventana de 10 segundos
metricas_cliente = (stream_practica
    .withWatermark("timestamp", "5 seconds")
    .groupBy(
        window(col("timestamp"), "10 seconds"),
        col("cliente_id")
    )
    .agg(
        count("value").alias("num_tx"),
        spark_round(avg("monto"), 2).alias("monto_promedio")
    )
    # Detectar clientes con actividad inusual
    .withColumn("nivel_alerta",
        when(col("num_tx") > 5, "ALTA")    # más de 5 transacciones en 10s
        .when(col("monto_promedio") > 5000, "MEDIA")
        .otherwise("NORMAL")
    )
    .select(
        col("window.start").alias("ventana_inicio"),
        "cliente_id", "num_tx", "monto_promedio", "nivel_alerta"
    )
)

query_practica = (metricas_cliente.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "/tmp/u08_ckpt_practica/")
    .start()
)

time.sleep(25)
query_practica.stop()
print("Pipeline de alertas detenido")

**Resultado esperado:**
```
Batch: 2
+-------------------+-----------+------+--------------+------------+
|     ventana_inicio|cliente_id |num_tx|monto_promedio|nivel_alerta|
+-------------------+-----------+------+--------------+------------+
|2024-01-15 10:00:00|         0 |     6|       3214.50|        ALTA|
|2024-01-15 10:00:00|        15 |     3|       6821.33|       MEDIA|
|2024-01-15 10:00:00|        32 |     5|       2100.00|      NORMAL|
...
```

---
## Preguntas de revisión

1. ¿Cuál es la diferencia entre batch y streaming?
2. ¿Qué es una tumbling window y cómo difiere de una sliding window?
3. ¿Para qué sirve el watermark en un streaming job?
4. ¿Qué es el `checkpointLocation` y por qué es obligatorio en producción?
5. ¿En qué casos usarías `outputMode("complete")` vs `outputMode("append")`?
6. ¿Qué ventaja tiene `foreachBatch` sobre un sink estándar?